# Notebook 02: Evaluasi RAG + Query Rewriting — PubMedQA

Konfigurasi **RAG + Query Rewriting (QR)** menggunakan BM25 sebagai retriever.
Query Rewriting mereformulasi pertanyaan asli agar lebih kaya terminologi medis sebelum retrieval.

## Konfigurasi
| Parameter | Nilai |
|-----------|-------|
| Query Rewriting | **Ya** (LLM reformulasi query) |
| Context Reranking | Tidak |
| Active Detection | Tidak |
| Retriever | BM25 (rank\_bm25) |
| LLM | llama3.2 via Ollama |
| Embedding | nomic-embed-text (hanya untuk RAGAS Answer Relevancy) |
| Dataset | PubMedQA pqa\_labeled |
| Sampel | 500 |

## Alur
1. **DEMO** 5 sampel (verifikasi pipeline + contoh query rewriting)
2. **Fase 1** Generate 500 jawaban + label accuracy (cepat)
3. **Fase 2** Evaluasi RAGAS per sampel (lambat, bisa semalam)
4. **Analisis + Perbandingan** Ringkasan metrik vs Baseline

## 1. Impor Library

In [2]:
import os, sys, json, pickle, time, re, warnings
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Dict, Tuple
from pathlib import Path
from datetime import datetime
from collections import Counter

from rank_bm25 import BM25Okapi
import ollama
from datasets import load_dataset

warnings.filterwarnings('ignore', category=DeprecationWarning)
from ragas import EvaluationDataset, SingleTurnSample, evaluate, RunConfig
from ragas.metrics import (
    _Faithfulness,
    _ResponseRelevancy,
    _LLMContextPrecisionWithReference,
    _LLMContextRecall
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_ollama import OllamaLLM, OllamaEmbeddings

print('Semua library berhasil diimpor!')
print(f'Python: {sys.version.split()[0]} | NumPy: {np.__version__} | Pandas: {pd.__version__}')

C:\Users\Ricky Wijaya\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Semua library berhasil diimpor!
Python: 3.11.9 | NumPy: 2.3.5 | Pandas: 2.3.3


## 2. Konfigurasi

In [6]:
LLM_MODEL   = 'llama3.2'
EMBED_MODEL = 'nomic-embed-text'  # Hanya untuk RAGAS

TOP_K_RETRIEVAL = 5

DATASET_NAME   = 'qiaojin/PubMedQA'
DATASET_SUBSET = 'pqa_labeled'

MAX_SAMPLES = 500

TEMPERATURE = 0.0
SEED        = 42

NOTEBOOK_DIR    = Path('.')
BM25_INDEX_PATH = NOTEBOOK_DIR / 'pubmedqa_bm25.pkl'  # Shared dengan baseline
RESULTS_DIR     = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)

CONFIG_NAME    = 'qr'
PHASE1_PATH    = RESULTS_DIR / f'{CONFIG_NAME}_phase1_answers.json'
PHASE2_PATH    = RESULTS_DIR / f'{CONFIG_NAME}_phase2_ragas.json'
FINAL_CSV_PATH = RESULTS_DIR / f'{CONFIG_NAME}_results.csv'

BASELINE_PHASE1_PATH = RESULTS_DIR / 'baseline_phase1_answers.json'
BASELINE_PHASE2_PATH = RESULTS_DIR / 'baseline_phase2_ragas.json'

print('Konfigurasi:')
print(f'  LLM        : {LLM_MODEL}')
print(f'  Embedding  : {EMBED_MODEL} (RAGAS only)')
print(f'  Retriever  : BM25 + Query Rewriting')
print(f'  Top-K      : {TOP_K_RETRIEVAL}')
print(f'  Sampel     : {MAX_SAMPLES}')
print(f'  Config     : {CONFIG_NAME}')

Konfigurasi:
  LLM        : llama3.2
  Embedding  : nomic-embed-text (RAGAS only)
  Retriever  : BM25 + Query Rewriting
  Top-K      : 5
  Sampel     : 500
  Config     : qr


## 3. Data Classes dan Tokenizer BM25

In [7]:
@dataclass
class Document:
    text         : str
    pubid        : str
    question     : str
    section_label: str
    answer       : str
    decision     : str

@dataclass
class RetrievalResult:
    document: Document
    score   : float


def tokenize_bm25(text: str) -> List[str]:
    """Tokenizer untuk BM25: hapus tanda baca, lowercase, split spasi."""
    return re.sub(r'[^a-zA-Z0-9\s]', ' ', text.lower()).split()


sample_text = 'Does aspirin (75mg) reduce myocardial infarction risk?'
print(f'Tokenisasi BM25: {tokenize_bm25(sample_text)}')
print('Data classes dan tokenizer siap.')

Tokenisasi BM25: ['does', 'aspirin', '75mg', 'reduce', 'myocardial', 'infarction', 'risk']
Data classes dan tokenizer siap.


## 4. Muat Dataset dan Bangun BM25 Index

BM25 index di-share dengan notebook baseline — gunakan file yang sama agar tidak rebuild dari scratch.

In [8]:
def load_pubmedqa(subset=DATASET_SUBSET, max_samples=MAX_SAMPLES):
    print(f'Memuat PubMedQA ({subset})...')
    dataset = load_dataset(DATASET_NAME, subset, trust_remote_code=True)
    data    = dataset['train']
    if max_samples and len(data) > max_samples:
        data = data.select(range(max_samples))
    print(f'Dimuat {len(data)} sampel')
    return data


def prepare_documents(data) -> List[Document]:
    docs = []
    for item in data:
        pubid = str(item['pubid'])
        for ctx, label in zip(item['context']['contexts'], item['context']['labels']):
            docs.append(Document(
                text=ctx.strip(), pubid=pubid,
                question=item['question'], section_label=label,
                answer=item['long_answer'], decision=item['final_decision']
            ))
    print(f'Total potongan dokumen: {len(docs)}')
    return docs


def load_or_build_bm25(data) -> Tuple[BM25Okapi, List[Document]]:
    """Muat BM25 index dari file jika ada (shared dengan baseline), atau bangun dari scratch."""
    if BM25_INDEX_PATH.exists():
        print(f'Memuat BM25 index dari {BM25_INDEX_PATH}...')
        with open(BM25_INDEX_PATH, 'rb') as f:
            saved = pickle.load(f)
        print(f'Dimuat: {len(saved["documents"])} dokumen')
        return saved['bm25'], saved['documents']
    else:
        print('Membangun BM25 index...')
        documents = prepare_documents(data)
        tokenized = [tokenize_bm25(d.text) for d in documents]
        bm25      = BM25Okapi(tokenized)
        with open(BM25_INDEX_PATH, 'wb') as f:
            pickle.dump({'bm25': bm25, 'documents': documents}, f)
        print(f'Index disimpan ke {BM25_INDEX_PATH}')
        return bm25, documents


t0 = time.time()
pubmedqa_data         = load_pubmedqa()
bm25_index, documents = load_or_build_bm25(pubmedqa_data)
print(f'Selesai dalam {time.time()-t0:.1f} detik')

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'qiaojin/PubMedQA' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Memuat PubMedQA (pqa_labeled)...
Dimuat 500 sampel
Memuat BM25 index dari pubmedqa_bm25.pkl...
Dimuat: 1706 dokumen
Selesai dalam 3.1 detik


## 5. Query Rewriting dengan LLM

LLM mereformulasi query asli menjadi pertanyaan yang lebih kaya terminologi medis untuk meningkatkan kualitas BM25 retrieval.

**Kenapa QR membantu BM25?**
- BM25 bergantung pada keyword matching — query yang lebih spesifik dan kaya terminologi meningkatkan recall dokumen relevan.
- LLM dapat mengekspansi singkatan, menambahkan sinonim medis, dan memperjelas konteks pertanyaan.

In [9]:
QUERY_REWRITE_PROMPT = (
    'You are a query rewriting assistant for a biomedical question-answering system.\n'
    'Rewrite the following medical question to improve retrieval from a PubMed research database.\n\n'
    'Rules:\n'
    '1. Be more specific and add relevant medical/scientific terminology.\n'
    '2. Expand abbreviations (e.g. "MI" -> "myocardial infarction").\n'
    '3. Preserve the original yes/no/maybe answerable intent.\n'
    '4. Output ONLY the rewritten question, no explanations.\n\n'
    'Original question: {query}\n\n'
    'Rewritten question:'
)


def rewrite_query(query: str) -> str:
    """
    Mereformulasi query menggunakan LLM untuk meningkatkan kualitas BM25 retrieval.
    Jika LLM gagal atau output terlalu pendek, kembalikan query asli.
    """
    prompt = QUERY_REWRITE_PROMPT.format(query=query)
    try:
        response = ollama.generate(
            model=LLM_MODEL,
            prompt=prompt,
            options={'temperature': 0.3, 'seed': SEED, 'num_predict': 150}
        )
        rewritten = response['response'].strip().replace('\n', ' ')
        return rewritten if len(rewritten) >= 10 else query
    except Exception as e:
        print(f'  [QR Error] {e} -- menggunakan query asli')
        return query


# Test
test_queries_qr = [
    'Does aspirin reduce the risk of myocardial infarction?',
    'Is vitamin D effective for COVID-19?',
    'Can exercise prevent T2DM?',
]

print('Contoh Query Rewriting:')
print('=' * 70)
for q in test_queries_qr:
    rw = rewrite_query(q)
    print(f'\nAsli    : {q}')
    print(f'Rewrite : {rw}')

Contoh Query Rewriting:

Asli    : Does aspirin reduce the risk of myocardial infarction?
Rewrite : Does low-dose aspirin therapy significantly decrease the incidence of recurrent myocardial infarction in patients with established coronary artery disease?

Asli    : Is vitamin D effective for COVID-19?
Rewrite : Is vitamin D supplementation effective in reducing mortality and morbidity in patients with COVID-19?

Asli    : Can exercise prevent T2DM?
Rewrite : Can regular aerobic exercise interventions effectively reduce or prevent the onset of type 2 diabetes mellitus in adults?


## 6. Fungsi Retrieval dengan Query Rewriting (BM25)

In [10]:
def retrieve_with_qr(query: str, k: int = TOP_K_RETRIEVAL) -> Tuple[List[RetrievalResult], str]:
    """
    Retrieval menggunakan BM25 dengan Query Rewriting.
    Returns: (retrieved_docs, rewritten_query)
    """
    rewritten = rewrite_query(query)
    tokens    = tokenize_bm25(rewritten)
    scores    = bm25_index.get_scores(tokens)
    top_k     = np.argsort(scores)[::-1][:k]
    results   = [RetrievalResult(document=documents[i], score=float(scores[i])) for i in top_k]
    return results, rewritten


# Test
test_q = 'Does aspirin reduce the risk of myocardial infarction?'
test_r, test_rw = retrieve_with_qr(test_q)
print(f'Query asli : {test_q}')
print(f'Rewrite    : {test_rw}')
print(f'\nTop-{TOP_K_RETRIEVAL} dokumen (BM25 + QR):')
for i, r in enumerate(test_r, 1):
    print(f'  [{i}] Score={r.score:.4f} | {r.document.section_label} | {r.document.text[:90]}...')

Query asli : Does aspirin reduce the risk of myocardial infarction?
Rewrite    : Does low-dose aspirin therapy significantly decrease the incidence of recurrent myocardial infarction in patients with established coronary artery disease?

Top-5 dokumen (BM25 + QR):
  [1] Score=27.5966 | DESIGN | Within a prospective, population-based cohort study individuals without history of myocard...
  [2] Score=23.1007 | RESULTS | Of the 601 patients with cardiogenic shock, 287 (47.8%) were admitted to hospitals without...
  [3] Score=22.9104 | OBJECTIVE | To examine the effect of a weekend hospitalization on the timing and incidence of intensiv...
  [4] Score=22.2804 | BACKGROUND | The role of early revascularization among patients with acute myocardial infarction compli...
  [5] Score=21.5813 | METHODS | Of the 9681 women and 8888 men who attended risk assessment from 1967-1991, with follow-up...


## 7. Prompt Generasi dan Fungsi Generate

In [11]:
GENERATION_PROMPT = (
    'You are a medical research assistant. '
    'Answer a biomedical yes/no/maybe question based solely on the provided scientific abstracts.\n\n'
    'Context from medical literature:\n{context}\n\n'
    'Question: {question}\n\n'
    'Instructions:\n'
    '- Carefully read the context and assess whether it supports or refutes the question.\n'
    '- Provide a brief explanation (2-3 sentences) using ONLY the information above.\n'
    '- End your response with EXACTLY ONE of these words on its own line: yes, no, or maybe.\n'
    '  - yes   : the evidence supports the hypothesis, even if not perfectly conclusive\n'
    '  - no    : the evidence refutes or does not support the hypothesis\n'
    '  - maybe : ONLY if the evidence is directly contradictory (some findings say yes,\n'
    '            others say no), or if the context contains no relevant information at all\n'
    '- IMPORTANT: If the evidence leans in one direction, even partially, choose yes or no.\n'
    '  Do NOT use maybe simply because the evidence is limited or not 100%% certain.\n\n'
    'Answer:'
)


def generate_answer(query: str, retrieved: List[RetrievalResult]) -> str:
    context = '\n\n'.join(
        f'[{i}] ({r.document.section_label}): {r.document.text}'
        for i, r in enumerate(retrieved, 1)
    )
    response = ollama.generate(
        model=LLM_MODEL,
        prompt=GENERATION_PROMPT.format(context=context, question=query),
        options={'temperature': TEMPERATURE, 'seed': SEED, 'num_predict': 300}
    )
    return response['response'].strip()


test_ans = generate_answer(test_q, test_r)
print('Output generation:')
print('-' * 60)
print(test_ans)

Output generation:
------------------------------------------------------------
Based on the provided scientific abstracts, there is no direct evidence to support or refute the question of whether aspirin reduces the risk of myocardial infarction. The context focuses on the impact of hospital revascularization services and weekend hospitalizations on patient outcomes, but does not mention aspirin or its effects on myocardial infarction risk.

maybe


## 8. Ekstraksi Label yes/no/maybe

In [8]:
def extract_label(answer: str) -> str:
    """
    Ekstrak prediksi yes/no/maybe dari teks jawaban.
    Strategi (berurutan hingga ditemukan):
      1. Kata standalone di 3 baris terakhir (non-kosong)
      2. Kata standalone di seluruh teks
      3. Default ke 'maybe'
    """
    lines = [l.strip().lower() for l in answer.split('\n') if l.strip()]
    for line in reversed(lines[-3:]):
        word = re.sub(r'[^a-z]', '', line)
        if word in ('yes', 'no', 'maybe'):
            return word
    for label in ('yes', 'no', 'maybe'):
        if re.search(r'\b' + label + r'\b', answer.lower()):
            return label
    return 'maybe'


cases = [
    ('Strong evidence.\nyes',    'yes'),
    ('No effect found.\nno',     'no'),
    ('Mixed results.\nmaybe',    'maybe'),
    ('Verdict: yes.',             'yes'),
    ('Totally unclear.',          'maybe'),
]
print('Unit test extract_label:')
all_ok = True
for txt, exp in cases:
    pred = extract_label(txt)
    ok   = pred == exp
    all_ok = all_ok and ok
    print(f'  [{"PASS" if ok else "FAIL"}] pred={pred!r} expected={exp!r}')
print(f'\nSemua lulus: {all_ok}')
print(f'Label dari test answer: {extract_label(test_ans)!r}')

Unit test extract_label:
  [PASS] pred='yes' expected='yes'
  [PASS] pred='no' expected='no'
  [PASS] pred='maybe' expected='maybe'
  [PASS] pred='yes' expected='yes'
  [PASS] pred='maybe' expected='maybe'

Semua lulus: True
Label dari test answer: 'maybe'


## 9. Setup RAGAS dengan Ollama

Sama persis dengan notebook baseline: `max_workers=1` untuk mencegah TimeoutError pada Ollama lokal.

In [9]:
ragas_llm = LangchainLLMWrapper(OllamaLLM(model=LLM_MODEL, temperature=0))
ragas_emb = LangchainEmbeddingsWrapper(OllamaEmbeddings(model=EMBED_MODEL))

ragas_run_config = RunConfig(
    timeout     = 300,
    max_retries = 2,
    max_wait    = 60,
    max_workers = 1,    # FIX TimeoutError: eksekusi sekuensial
    log_tenacity= False,
)

faithfulness_m   = _Faithfulness(llm=ragas_llm)
ans_relevancy_m  = _ResponseRelevancy(llm=ragas_llm, embeddings=ragas_emb)
ctx_precision_m  = _LLMContextPrecisionWithReference(llm=ragas_llm)
ctx_recall_m     = _LLMContextRecall(llm=ragas_llm)

ALL_METRICS = [faithfulness_m, ans_relevancy_m, ctx_precision_m, ctx_recall_m]

print('RAGAS siap (tanpa OpenAI, max_workers=1):')
for m in ALL_METRICS:
    print(f'  - {m.name}')


def evaluate_ragas_single(question: str, answer: str,
                          contexts: List[str], reference: str) -> Dict:
    """Evaluasi RAGAS untuk 1 sampel. Mengembalikan NaN jika gagal."""
    nan = float('nan')
    sample  = SingleTurnSample(
        user_input=question, response=answer,
        retrieved_contexts=contexts, reference=reference
    )
    dataset = EvaluationDataset(samples=[sample])
    try:
        result = evaluate(
            dataset, metrics=ALL_METRICS,
            llm=ragas_llm, embeddings=ragas_emb,
            run_config=ragas_run_config,
            raise_exceptions=False, show_progress=False
        )
        row = result.to_pandas().iloc[0]
        return {
            'faithfulness'      : float(row.get('faithfulness',                         nan)),
            'answer_relevancy'  : float(row.get('answer_relevancy',                     nan)),
            'context_precision' : float(row.get('llm_context_precision_with_reference', nan)),
            'context_recall'    : float(row.get('context_recall',                       nan)),
        }
    except Exception as e:
        print(f'  [RAGAS Error] {type(e).__name__}: {e}')
        return {'faithfulness': nan, 'answer_relevancy': nan,
                'context_precision': nan, 'context_recall': nan}

RAGAS siap (tanpa OpenAI, max_workers=1):
  - faithfulness
  - answer_relevancy
  - llm_context_precision_with_reference
  - context_recall


---
## DEMO: Uji Coba 5 Sampel

Verifikasi pipeline QR berjalan dengan benar. Tampilkan query asli vs rewritten.

> Estimasi: ~3-10 menit (tergantung kecepatan Ollama)

In [10]:
DEMO_SIZE    = 5
demo_results = []

print(f'DEMO: {DEMO_SIZE} sampel pertama (RAG + Query Rewriting)')
print('=' * 70)

for i in range(DEMO_SIZE):
    s         = pubmedqa_data[i]
    q         = s['question']
    gt        = s['final_decision']
    ref       = s['long_answer']

    print(f'\n[{i+1}/{DEMO_SIZE}] PubID: {s["pubid"]}')
    print(f'  Q asli   : {q[:75]}...')
    print(f'  GT       : {gt}')

    t0 = time.time()
    retrieved, rewritten = retrieve_with_qr(q)
    t_ret = time.time() - t0
    print(f'  Q rewrite: {rewritten[:75]}...')

    t0     = time.time()
    answer = generate_answer(q, retrieved)
    t_gen  = time.time() - t0

    predicted = extract_label(answer)
    correct   = predicted == gt

    t0      = time.time()
    ctxs    = [r.document.text for r in retrieved]
    scores  = evaluate_ragas_single(q, answer, ctxs, ref)
    t_ragas = time.time() - t0

    demo_results.append({
        'idx': i, 'pubid': str(s['pubid']), 'question': q,
        'rewritten_query': rewritten,
        'ground_truth': gt, 'predicted_label': predicted,
        'is_correct': correct, 'answer': answer,
        'contexts': ctxs, 'reference': ref, **scores,
        't_retrieval': round(t_ret,2), 't_generation': round(t_gen,2), 't_ragas': round(t_ragas,2)
    })

    verdict = 'BENAR' if correct else 'SALAH'
    print(f'  Pred     : {predicted} [{verdict}]')
    print(f'  RAGAS    : faith={scores["faithfulness"]:.3f} | '
          f'rel={scores["answer_relevancy"]:.3f} | '
          f'cp={scores["context_precision"]:.3f} | '
          f'cr={scores["context_recall"]:.3f}')
    print(f'  Waktu    : ret={t_ret:.1f}s gen={t_gen:.1f}s ragas={t_ragas:.1f}s')

n_ok = sum(r['is_correct'] for r in demo_results)
print(f'\n{"="*70}')
print(f'RINGKASAN DEMO ({DEMO_SIZE} sampel -- RAG + QR):')
print(f'  Label Accuracy    : {n_ok}/{DEMO_SIZE} = {n_ok/DEMO_SIZE:.1%}')
print(f'  Hallucination Rate: {(DEMO_SIZE-n_ok)/DEMO_SIZE:.1%}')
for col in ['faithfulness','answer_relevancy','context_precision','context_recall']:
    print(f'  Avg {col:<22}: {np.nanmean([r[col] for r in demo_results]):.3f}')
print(f'{"="*70}')

DEMO: 5 sampel pertama (RAG + Query Rewriting)

[1/5] PubID: 21645374
  Q asli   : Do mitochondria play a role in remodelling lace plant leaves during program...
  GT       : yes
  Q rewrite: Do mitochondrial function and dynamics contribute to programmed cell death-...


Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\Ricky Wijaya\AppData\Local\Programs\Python\Python311\Lib\asyncio\events.py", line 84, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x0000019427C36100> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\Ricky Wijaya\AppData\Local\Programs\Python\Python311\Lib\asyncio\events.py", line 84, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x0000019427C36100> is already entered
Task was destroyed but it is pending!
task: <Task pending name='Task-70' coro=<_async_in_context.<locals>.run_in_context() running at C:\Users\Ricky Wijaya\AppData\Local\Programs\Python\Python311\Lib\site-packages\ipykernel\utils.py:60> wait_for=<Task pendin

  Pred     : yes [BENAR]
  RAGAS    : faith=1.000 | rel=0.897 | cp=nan | cr=0.750
  Waktu    : ret=8.2s gen=61.6s ragas=703.5s

[2/5] PubID: 16418930
  Q asli   : Landolt C and snellen e acuity: differences in strabismus amblyopia?...
  GT       : no
  Q rewrite: Landolt C and Snellen E acuity: a comparative analysis of visual field defe...
  Pred     : yes [SALAH]
  RAGAS    : faith=0.556 | rel=0.675 | cp=1.000 | cr=0.667
  Waktu    : ret=8.3s gen=62.1s ragas=607.9s

[3/5] PubID: 9488747
  Q asli   : Syncope during bathing in infants, a pediatric form of water-induced urtica...
  GT       : yes
  Q rewrite: What is the prevalence and clinical characteristics of infantile water-indu...


Exception raised in Job[1]: OutputParserException(Invalid json output: Given input:
{
    "response": "The question of syncope during bathing in infants being a pediatric form of water-induced urticaria is supported by the context. The case reports describe eight infants who experienced pale, hypotonic, and unreactive states after immersion in water, which recovered quickly upon withdrawal from the bath and stimulation. Additionally, an increase in blood histamine levels was found in two tested infants, suggesting a possible link to histamine release.\n\nyes"
}

Generated question:
"Is syncope during bathing in infants a pediatric form of water-induced urticaria?"

Noncommittal answer: 1

Output in JSON format:
{
    "question": "Is syncope during bathing in infants a pediatric form of water-induced urticaria?",
    "noncommittal": 1
}
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE )
Exception raised in Job[2]: TimeoutError()


  Pred     : yes [BENAR]
  RAGAS    : faith=1.000 | rel=nan | cp=nan | cr=1.000
  Waktu    : ret=8.1s gen=43.9s ragas=774.3s

[4/5] PubID: 17208539
  Q asli   : Are the long-term results of the transanal pull-through equal to those of t...
  GT       : no
  Q rewrite: Are the long-term outcomes and complications of transanal pull-through proc...


Exception raised in Job[1]: OutputParserException(Invalid json output: The output string did not satisfy the constraints given in the prompt. Fix the output string and return it.
Please return the output in a JSON format that complies with the following schema as specified in JSON Schema:
{
    "properties": {
        "text": {
            "title": "Text",
            "type": "string"
        }
    },
    "required": ["text"],
    "title": "StringIO",
    "type": "object"
}

-----------------------------

Now perform the same with the following input
input: {
    "output_string": "Given answer is \\
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE )


  Pred     : yes [SALAH]
  RAGAS    : faith=0.833 | rel=nan | cp=0.887 | cr=0.750
  Waktu    : ret=10.8s gen=59.3s ragas=746.1s

[5/5] PubID: 10808977
  Q asli   : Can tailored interventions increase mammography use among HMO women?...
  GT       : yes
  Q rewrite: Can targeted behavioral interventions incorporating patient education and r...


Exception raised in Job[0]: TimeoutError()
Exception raised in Job[1]: TimeoutError()
Exception raised in Job[2]: TimeoutError()


  Pred     : yes [BENAR]
  RAGAS    : faith=nan | rel=nan | cp=nan | cr=1.000
  Waktu    : ret=8.8s gen=41.4s ragas=1128.4s

RINGKASAN DEMO (5 sampel -- RAG + QR):
  Label Accuracy    : 3/5 = 60.0%
  Hallucination Rate: 40.0%
  Avg faithfulness          : 0.847
  Avg answer_relevancy      : 0.786
  Avg context_precision     : 0.944
  Avg context_recall        : 0.833


---
## Fase 1: Generate Semua Jawaban (500 Sampel)

Retrieval + Query Rewriting + Generation. Rewritten query disimpan untuk analisis.
Disimpan inkremental setiap 10 sampel — aman untuk resume.

> Estimasi: ~35-70 menit untuk 500 sampel (sedikit lebih lambat dari baseline karena overhead QR per sampel)

In [11]:
if PHASE1_PATH.exists():
    with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
        phase1_results = json.load(f)['results']
    start_from = len(phase1_results)
    print(f'Resume Fase 1: {start_from}/{MAX_SAMPLES} sudah selesai.')
else:
    phase1_results, start_from = [], 0
    print(f'Memulai Fase 1: {MAX_SAMPLES} sampel.')

if start_from < MAX_SAMPLES:
    print(f'Memproses {MAX_SAMPLES - start_from} sampel tersisa...\n')
    t_start = time.time()
    for i in range(start_from, MAX_SAMPLES):
        s          = pubmedqa_data[i]
        q, gt, ref = s['question'], s['final_decision'], s['long_answer']
        retrieved, rewritten = retrieve_with_qr(q)
        answer     = generate_answer(q, retrieved)
        predicted  = extract_label(answer)
        phase1_results.append({
            'idx': i, 'pubid': str(s['pubid']), 'question': q,
            'rewritten_query': rewritten,
            'ground_truth': gt, 'predicted_label': predicted,
            'is_correct': predicted == gt, 'answer': answer,
            'contexts': [r.document.text for r in retrieved],
            'reference': ref, 'retrieval_scores': [r.score for r in retrieved],
        })
        if (i + 1) % 10 == 0 or i == MAX_SAMPLES - 1:
            with open(PHASE1_PATH, 'w', encoding='utf-8') as f:
                json.dump({'config': CONFIG_NAME,
                           'timestamp': datetime.now().isoformat(),
                           'max_samples': MAX_SAMPLES, 'completed': i+1,
                           'results': phase1_results}, f, indent=2, ensure_ascii=False)
            done = i + 1
            acc  = sum(r['is_correct'] for r in phase1_results) / done
            eta  = (time.time()-t_start) / done * (MAX_SAMPLES-done) / 60
            print(f'  [{done:3d}/{MAX_SAMPLES}] Akurasi: {acc:.1%} | pred={predicted}, gt={gt} | ETA {eta:.1f} mnt')
    print(f'\nFase 1 selesai! Disimpan ke {PHASE1_PATH}')
else:
    print(f'Fase 1 sudah selesai ({MAX_SAMPLES} sampel).')

Resume Fase 1: 90/500 sudah selesai.
Memproses 410 sampel tersisa...

  [100/500] Akurasi: 58.0% | pred=yes, gt=yes | ETA 41.7 mnt
  [110/500] Akurasi: 57.3% | pred=yes, gt=yes | ETA 70.7 mnt
  [120/500] Akurasi: 55.8% | pred=yes, gt=yes | ETA 92.3 mnt
  [130/500] Akurasi: 53.8% | pred=yes, gt=maybe | ETA 110.4 mnt
  [140/500] Akurasi: 54.3% | pred=yes, gt=yes | ETA 125.5 mnt
  [150/500] Akurasi: 53.3% | pred=yes, gt=no | ETA 133.4 mnt
  [160/500] Akurasi: 52.5% | pred=yes, gt=no | ETA 142.0 mnt
  [170/500] Akurasi: 51.2% | pred=yes, gt=maybe | ETA 147.6 mnt
  [180/500] Akurasi: 51.1% | pred=yes, gt=no | ETA 151.2 mnt
  [190/500] Akurasi: 51.1% | pred=maybe, gt=no | ETA 154.1 mnt
  [200/500] Akurasi: 50.0% | pred=yes, gt=yes | ETA 155.4 mnt
  [210/500] Akurasi: 50.5% | pred=yes, gt=yes | ETA 156.2 mnt
  [220/500] Akurasi: 50.9% | pred=yes, gt=yes | ETA 156.0 mnt
  [230/500] Akurasi: 49.6% | pred=yes, gt=no | ETA 154.5 mnt
  [240/500] Akurasi: 50.4% | pred=yes, gt=yes | ETA 153.0 mnt
  

### Analisis Fase 1 (Label Accuracy)

In [12]:
with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    results_p1 = json.load(f)['results']
n         = len(results_p1)
n_correct = sum(r['is_correct'] for r in results_p1)
gts       = [r['ground_truth']    for r in results_p1]
preds     = [r['predicted_label'] for r in results_p1]

print(f'ANALISIS FASE 1 -- {n} sampel (RAG + Query Rewriting)')
print('=' * 55)
print(f'Label Accuracy    : {n_correct}/{n} = {n_correct/n:.1%}')
print(f'Hallucination Rate: {(n-n_correct)/n:.1%}\n')
print(f'  {"Label":<8} | {"Ground Truth":>12} | {"Prediksi":>12}')
print(f'  {"-"*8}-+{"-"*14}-+{"-"*12}')
for lbl in ['yes','no','maybe']:
    g, p = gts.count(lbl), preds.count(lbl)
    print(f'  {lbl:<8} | {g:>10} ({g/n:.0%}) | {p:>10} ({p/n:.0%})')
print('\nConfusion Matrix (baris=GT, kolom=Prediksi):')
lbls = ['yes','no','maybe']
print('  ' + f'{"GT/Pred":>8}' + ''.join(f'{l:>8}' for l in lbls))
for gt_l in lbls:
    row = f'  {gt_l:>8}'
    for pr_l in lbls:
        cnt = sum(1 for r in results_p1 if r['ground_truth']==gt_l and r['predicted_label']==pr_l)
        row += f'{cnt:>8}'
    print(row)

ANALISIS FASE 1 -- 500 sampel (RAG + Query Rewriting)
Label Accuracy    : 250/500 = 50.0%
Hallucination Rate: 50.0%

  Label    | Ground Truth |     Prediksi
  ---------+---------------+------------
  yes      |        275 (55%) |        406 (81%)
  no       |        159 (32%) |         66 (13%)
  maybe    |         66 (13%) |         28 (6%)

Confusion Matrix (baris=GT, kolom=Prediksi):
   GT/Pred     yes      no   maybe
       yes     227      37      11
        no     123      21      15
     maybe      56       8       2


---
## Fase 2: Evaluasi RAGAS

Jalankan setelah Fase 1 selesai. Mendukung resume — aman dihentikan kapan saja.

> Estimasi: ~2-5 menit per sampel. Disarankan dijalankan semalam.

In [1]:
if PHASE2_PATH.exists():
    with open(PHASE2_PATH, 'r', encoding='utf-8') as f:
        phase2_results = json.load(f)['results']
    done_idx = {r['idx'] for r in phase2_results}
    print(f'Resume Fase 2: {len(done_idx)}/{len(results_p1)} sudah dievaluasi.')
else:
    phase2_results, done_idx = [], set()
    print(f'Memulai Fase 2: {len(results_p1)} sampel.')

remaining = [r for r in results_p1 if r['idx'] not in done_idx]
print(f'Sisa: {len(remaining)} sampel | Estimasi ~{len(remaining)*3/60:.1f} jam\n')

t_p2 = time.time()
for i, r in enumerate(remaining):
    scores = evaluate_ragas_single(r['question'], r['answer'], r['contexts'], r['reference'])
    phase2_results.append({
        'idx': r['idx'], 'ground_truth': r['ground_truth'],
        'predicted_label': r['predicted_label'], 'is_correct': r['is_correct'],
        **scores
    })
    if (i + 1) % 5 == 0 or i == len(remaining) - 1:
        with open(PHASE2_PATH, 'w', encoding='utf-8') as f:
            json.dump({'config': CONFIG_NAME, 'timestamp': datetime.now().isoformat(),
                       'results': phase2_results}, f, indent=2, ensure_ascii=False)
        done  = i + 1
        total = len(remaining)
        eta   = (time.time()-t_p2)/done*(total-done)/60 if done < total else 0
        avg_f = np.nanmean([x['faithfulness'] for x in phase2_results])
        print(f'  [{done:3d}/{total}] idx={r["idx"]} | '
              f'faith={scores["faithfulness"]:.3f} | avg_f={avg_f:.3f} | ETA {eta:.1f} mnt')
print(f'\nFase 2 selesai! Disimpan ke {PHASE2_PATH}')

NameError: name 'PHASE2_PATH' is not defined

---
## Hasil Akhir: Gabungkan Fase 1 + Fase 2 -> CSV

In [ ]:
with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    results_p1 = json.load(f)['results']
with open(PHASE2_PATH, 'r', encoding='utf-8') as f:
    results_p2 = json.load(f)['results']

p2_lookup = {r['idx']: r for r in results_p2}

df = pd.DataFrame([{
    'idx': r['idx'], 'pubid': r['pubid'], 'question': r['question'],
    'rewritten_query': r.get('rewritten_query', ''),
    'ground_truth': r['ground_truth'], 'predicted_label': r['predicted_label'],
    'is_correct': r['is_correct'],
    'faithfulness'      : p2_lookup.get(r['idx'],{}).get('faithfulness',       float('nan')),
    'answer_relevancy'  : p2_lookup.get(r['idx'],{}).get('answer_relevancy',   float('nan')),
    'context_precision' : p2_lookup.get(r['idx'],{}).get('context_precision',  float('nan')),
    'context_recall'    : p2_lookup.get(r['idx'],{}).get('context_recall',     float('nan')),
} for r in results_p1])

df.to_csv(FINAL_CSV_PATH, index=False, encoding='utf-8')
print(f'Disimpan ke: {FINAL_CSV_PATH}')
print(f'Total: {len(df)} | RAGAS lengkap: {df["faithfulness"].notna().sum()}')
df.head(10)

## Ringkasan Metrik Evaluasi RAG + Query Rewriting

In [ ]:
n, n_correct = len(df), int(df['is_correct'].sum())
label_acc  = n_correct / n
hallu_rate = 1 - label_acc
avg_f  = df['faithfulness'].mean()
avg_r  = df['answer_relevancy'].mean()
avg_cp = df['context_precision'].mean()
avg_cr = df['context_recall'].mean()

print('=' * 60)
print(f'  RAG + QUERY REWRITING (BM25+QR) -- {n} sampel')
print('=' * 60)
print(f'  {"Metrik":<32} {"Nilai":>10}')
print(f'  {"-"*42}')
print(f'  {"Label Accuracy":<32} {label_acc:>9.1%}')
print(f'  {"Hallucination Rate":<32} {hallu_rate:>9.1%}')
print(f'  {"-"*42}')
print(f'  {"Faithfulness (RAGAS)":<32} {avg_f:>10.4f}')
print(f'  {"Answer Relevancy (RAGAS)":<32} {avg_r:>10.4f}')
print(f'  {"Context Precision (RAGAS)":<32} {avg_cp:>10.4f}')
print(f'  {"Context Recall (RAGAS)":<32} {avg_cr:>10.4f}')
print('=' * 60)
print('\nPer-label accuracy:')
for lbl in ['yes','no','maybe']:
    sub = df[df['ground_truth']==lbl]
    if len(sub):
        print(f'  {lbl:>5}: {sub["is_correct"].mean():.1%} ({int(sub["is_correct"].sum())}/{len(sub)}) '
              f'| faithfulness={sub["faithfulness"].mean():.3f}')
print('\nBaris untuk tabel skripsi:')
print(f'  | BM25 + QR | {label_acc:.3f} | {hallu_rate:.3f} | '
      f'{avg_f:.3f} | {avg_r:.3f} | {avg_cp:.3f} | {avg_cr:.3f} |')

---
## Perbandingan: Baseline (BM25) vs BM25 + Query Rewriting

Jalankan setelah kedua notebook selesai pada Fase 1 dan Fase 2.

> Memuat hasil dari `../results/baseline_*.json` dan `../results/qr_*.json`

In [ ]:
if not BASELINE_PHASE1_PATH.exists() or not BASELINE_PHASE2_PATH.exists():
    print('File baseline belum tersedia. Jalankan notebook 03-RAG-Baseline-Evaluation.ipynb terlebih dahulu.')
else:
    with open(BASELINE_PHASE1_PATH, 'r', encoding='utf-8') as f:
        bl_p1 = json.load(f)['results']
    with open(BASELINE_PHASE2_PATH, 'r', encoding='utf-8') as f:
        bl_p2 = json.load(f)['results']
    bl_p2_lu = {r['idx']: r for r in bl_p2}

    df_bl = pd.DataFrame([{
        'idx': r['idx'], 'ground_truth': r['ground_truth'],
        'predicted_label': r['predicted_label'], 'is_correct': r['is_correct'],
        'faithfulness'     : bl_p2_lu.get(r['idx'],{}).get('faithfulness',       float('nan')),
        'answer_relevancy' : bl_p2_lu.get(r['idx'],{}).get('answer_relevancy',   float('nan')),
        'context_precision': bl_p2_lu.get(r['idx'],{}).get('context_precision',  float('nan')),
        'context_recall'   : bl_p2_lu.get(r['idx'],{}).get('context_recall',     float('nan')),
    } for r in bl_p1])

    configs = {
        'Baseline (BM25)' : df_bl,
        'BM25 + QR'       : df,
    }

    print('=' * 72)
    print('PERBANDINGAN: BASELINE vs QUERY REWRITING')
    print('=' * 72)
    hdr = f'  {"Konfigurasi":<22} | {"Accuracy":>9} | {"Hallu.":>6} | {"Faith.":>7} | {"Ans.Rel":>7} | {"Ctx.P":>6} | {"Ctx.R":>6}'
    print(hdr)
    print('  ' + '-' * 70)
    for cfg_name, dfc in configs.items():
        acc = dfc['is_correct'].mean()
        hr  = 1 - acc
        f_  = dfc['faithfulness'].mean()
        r_  = dfc['answer_relevancy'].mean()
        cp  = dfc['context_precision'].mean()
        cr  = dfc['context_recall'].mean()
        print(f'  {cfg_name:<22} | {acc:>8.1%} | {hr:>5.1%} | {f_:>7.4f} | {r_:>7.4f} | {cp:>6.4f} | {cr:>6.4f}')
    print('=' * 72)

    print('\nDelta (QR - Baseline):')
    metrics_cmp = [
        ('Label Accuracy',    df_bl['is_correct'].mean(),        df['is_correct'].mean()),
        ('Faithfulness',      df_bl['faithfulness'].mean(),      df['faithfulness'].mean()),
        ('Answer Relevancy',  df_bl['answer_relevancy'].mean(),  df['answer_relevancy'].mean()),
        ('Context Precision', df_bl['context_precision'].mean(), df['context_precision'].mean()),
        ('Context Recall',    df_bl['context_recall'].mean(),    df['context_recall'].mean()),
    ]
    for metrik, bl_val, qr_val in metrics_cmp:
        delta = qr_val - bl_val
        label = 'LEBIH BAIK' if delta > 0.001 else ('LEBIH BURUK' if delta < -0.001 else 'SAMA')
        print(f'  {metrik:<22}: {delta:+.4f}  ({label})')

    print('\nPer-label accuracy (Baseline vs QR):')
    for lbl in ['yes', 'no', 'maybe']:
        s_bl = df_bl[df_bl['ground_truth']==lbl]
        s_qr = df[df['ground_truth']==lbl]
        if len(s_bl) and len(s_qr):
            acc_bl = s_bl['is_correct'].mean()
            acc_qr = s_qr['is_correct'].mean()
            delta  = acc_qr - acc_bl
            print(f'  {lbl:>5}: Baseline={acc_bl:.1%}  QR={acc_qr:.1%}  Delta={delta:+.1%}')

---
## Analisis Kualitas Query Rewriting

Inspeksi kualitatif: cek contoh query asli vs rewritten dari Fase 1.

In [ ]:
with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    qr_p1 = json.load(f)['results']

print('Contoh Query Rewriting dari Fase 1:')
print('=' * 70)
for r in qr_p1[:10]:
    print(f'[{r["idx"]:3d}] Asli    : {r["question"][:75]}')
    print(f'       Rewrite : {r["rewritten_query"][:75]}')
    print(f'       GT={r["ground_truth"]} | Pred={r["predicted_label"]} | {"BENAR" if r["is_correct"] else "SALAH"}')
    print()

n_changed = sum(1 for r in qr_p1 if r['question'].strip().lower() != r['rewritten_query'].strip().lower())
print(f'Statistik QR:')
print(f'  Total sampel        : {len(qr_p1)}')
print(f'  Query berubah       : {n_changed} ({n_changed/len(qr_p1):.1%})')
print(f'  Query tidak berubah : {len(qr_p1)-n_changed} ({(len(qr_p1)-n_changed)/len(qr_p1):.1%})')